# 1. Imports & Environment Setup

All imports and global environment checks (device, seeds) live here.

In [1]:
from __future__ import absolute_import, division, print_function

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import argparse
import gc
import os
import time
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.init as init

from torch.utils.data import DataLoader, Subset, random_split
from torchvision import transforms
import torchvision.transforms.functional as F
import json

from sklearn.model_selection import train_test_split

import pandas as pd
from torch.utils.data import WeightedRandomSampler
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR

# 2. Dataset & Data Loading

Dataset classes, transforms, and dataloaders.

In [2]:
class FoodDataset(torch.utils.data.Dataset):
    def __init__(self, csv_path, train_image_path, transform):
        self.df = pd.read_csv(csv_path)
        self.img_dir = train_image_path
        self.transform = transform

    def __len__(self):
        return len(self.df)


    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row["img_name"]
        img_label = int(row["label"]-1)
        label_tensor = torch.tensor(img_label, dtype=torch.long)
        img_path = os.path.join(self.img_dir, img_name)
        img = Image.open(img_path).convert("RGB")
        tensor = self.transform(img)
        return tensor, label_tensor

class FoodTestDataset(torch.utils.data.Dataset):
    def __init__(self, test_img_path , transform =  transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406],
                             std=[0.229,0.224,0.225]),
])):
        self.img_dir = test_img_path
        self.img_names = sorted(os.listdir(self.img_dir))
        self.transform = transform

    def __len__(self):
        return len(self.img_names) 
    
    def __getitem__(self, idx):
        img_name = self.img_names[idx]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert("RGB")
        tensor = self.transform(image)
        return tensor, img_name


# 3. CNN Model Architecture

Definition of the CNN model.

In [3]:

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        b, c, _, _ = x.shape
        s = self.squeeze(x).view(b, c)
        e = self.excitation(s).view(b, c, 1, 1)
        return x * e
        
class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, use_batch_norm=False, use_se=False):
        super().__init__()
        layers = []
        self.use_se = use_se
        
        # conv 1
        conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=1, padding=1, bias=not use_batch_norm)
        init.kaiming_normal_(conv1.weight, mode="fan_in", nonlinearity="relu")
        layers.append(conv1)
        if use_batch_norm:
            layers.append(nn.BatchNorm2d(out_channels))
        layers.append(nn.ReLU())
        # conv 2
        conv2 = nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1, bias=not use_batch_norm)
        init.kaiming_normal_(conv2.weight, mode="fan_in", nonlinearity="relu")
        layers.append(conv2)
        if use_batch_norm:
            layers.append(nn.BatchNorm2d(out_channels))
        self.block = nn.Sequential(*layers)
        self.relu = nn.ReLU()
        if use_se:
            self.se = SEBlock(out_channels)
        
        if in_channels != out_channels:
            self.shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        out = self.block(x)
        if self.use_se:
            out = self.se(out)
        return self.relu(out + self.shortcut(x))

class CNN(nn.Module):
    def __init__(self, in_channels, n_hidden, n_classes, use_batch_norm=False, use_se=False):
        super().__init__()
        self.in_channels = in_channels
        self.n_hidden = n_hidden
        self.n_classes = n_classes

        layers = []
        input_dim = in_channels
        for out_channels in n_hidden:
            layers.append(ResBlock(input_dim, out_channels, use_batch_norm, use_se=use_se))
            layers.append(nn.MaxPool2d(2, stride=2))
            input_dim = out_channels

        layers.append(nn.AdaptiveAvgPool2d((1, 1)))
        layers.append(nn.Flatten())
        layers.append(nn.Dropout(p=0.4))

        lin = nn.Linear(input_dim, n_classes)
        init.kaiming_normal_(lin.weight, mode="fan_in")
        init.zeros_(lin.bias)
        layers.append(lin)

        self.layers = nn.Sequential(*layers)

    def forward(self, x):
        return self.layers(x)

    @property
    def device(self):
        return next(self.parameters()).device

# 4. Training Utilities

Helper functions: accuracy, evaluation, and submission generation.

In [4]:

def accuracy(predictions, targets):
    """
    Computes the prediction accuracy, i.e. the average of correct predictions
    of the network.
    
    Args:
      predictions: 2D float array of size [batch_size, n_classes], predictions of the model (logits)
      llabels: 1D int array of size [batch_size]. Ground truth labels for
               each sample in the batch
    Returns:
      accuracy: scalar float, the accuracy of predictions,
                i.e. the average correct predictions over the whole batch
    """

    pred = torch.argmax(predictions, axis=1)
    accuracy = (pred == targets).float().mean()
    
    return accuracy


def evaluate_model(model, data_loader, loss_module):
    """
    Performs the evaluation of the MLP model on a given dataset.

    Args:
      model: An instance of 'MLP', the model to evaluate.
      data_loader: The data loader of the dataset to evaluate.
    Returns:
      avg_accuracy: scalar float, the average accuracy of the model on the dataset.
    """

    model.eval()
    with torch.no_grad():
      correct = 0
      total = 0
      n_seen = 0
      running_loss = 0
      all_targets = []
      all_preds = []
      for x_batch, y_batch in data_loader:
            x_batch = x_batch.to(model.device)
            y_batch = y_batch.to(model.device)
            logits = model(x_batch)
            preds = torch.argmax(logits, axis=1)
            total += len(preds)
            correct += (preds == y_batch).float().sum()
            loss = loss_module(logits, y_batch)
            bs = y_batch.size(0)
            running_loss += loss.item() * bs
            n_seen += bs
            all_targets.append(y_batch.cpu())
            all_preds.append(preds.cpu())
        
      val_loss = running_loss / n_seen
      avg_accuracy = (correct / total).item()
      all_preds = torch.cat(all_preds, dim=0).numpy()
      all_targets = torch.cat(all_targets, dim=0).numpy()
        
    return avg_accuracy, val_loss, all_preds, all_targets

def test_model(model, data_loader, n_tta=10):
    model.eval()
    logits_accum = None
    all_names = []

    with torch.no_grad():
        for _ in range(n_tta):
            batch_logits = []
            batch_names = []
            for x_batch, name_batch in data_loader:
                x_batch = x_batch.to(model.device)
                logits = model(x_batch)
                batch_logits.append(logits)
                if _ == 0:
                    all_names.extend(name_batch)
            
            epoch_logits = torch.cat(batch_logits, dim=0)
            if logits_accum is None:
                logits_accum = epoch_logits
            else:
                logits_accum += epoch_logits

    preds = (torch.argmax(logits_accum, dim=1) + 1).cpu().tolist()
    df = pd.DataFrame({"img_name": all_names, "label": preds})
    df.to_csv("submission.csv", index=False)
    print("Saved submission.csv")


def save_results(logs):
    rows = []

    for epoch, metrics in logs.items():
        rows.append({
            "epoch": epoch,
            "train_acc": metrics[0],
            "train_loss": metrics[1],
            "val_acc": metrics[2],
            "val_loss": metrics[3],
        })
    df = pd.DataFrame(rows)
    os.makedirs("experiments", exist_ok=True)
    run_id = len([f for f in os.listdir("experiments") if f.startswith("run_")]) + 1
    df.to_csv(f"experiments/run_{run_id}.csv", index=False)
    return run_id
    
def mixup_batch(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0))
    mixed_x = lam * x + (1 - lam) * x[idx]
    return mixed_x, y, y[idx], lam

def cutmix_batch(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0))
    W, H = x.size(3), x.size(2)
    cut_w = int(W * np.sqrt(1 - lam))
    cut_h = int(H * np.sqrt(1 - lam))
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    mixed_x = x.clone()
    mixed_x[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam = 1 - (x2-x1)*(y2-y1) / (W*H)
    return mixed_x, y, y[idx], lam

# 5. Training Loop

Main training function with logging, early stopping, and scheduler hooks.

In [5]:
def train(hidden_dims, lr, use_batch_norm, batch_size, epochs, seed, val_frac, num_workers, csv_path, train_img_path, test_img_path, overfit_test, smal_test, use_scheduler, early_stopping, transform, weight_decay, use_se):
    """
    Performs a full training cycle of MLP model.

    Args:
      hidden_dims: A list of ints, specificying the hidden dimensionalities to use in the MLP.
      lr: Learning rate of the SGD to apply.
      use_batch_norm: If True, adds batch normalization layer into the network.
      batch_size: Minibatch size for the data loaders.
      epochs: Number of training epochs to perform.
      seed: Seed to use for reproducible results.
      data_dir: Directory where to store/find the CIFAR10 dataset.
    Returns:
      model: An instance of 'MLP', the trained model that performed best on the validation set.
      val_accuracies: A list of scalar floats, containing the accuracies of the model on the
                      validation set per epoch (element 0 - performance after epoch 1)
      test_accuracy: scalar float, average accuracy on the test dataset of the model that 
                     performed best on the validation.
      logging_dict: An arbitrary object containing logging information. This is for you to 
                    decide what to put in here.

    """

    # Set the random seeds for reproducibility
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():  # GPU operation have separate seed
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True
    g = torch.Generator().manual_seed(seed)
    
    # Set default device
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    print("DEVICE:", device, flush=True)


    # Datasets
    full_dataset = FoodDataset(csv_path, train_img_path, transform)
    
    val_dataset = FoodDataset(csv_path, train_img_path, transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225])]))

    test_dataset = FoodTestDataset(test_img_path)
    df_labels = pd.read_csv(csv_path)
    labels = np.array(df_labels["label"].values - 1)
    
    class_counts = np.bincount(labels)
    
    worst_classes = [27, 33, 34, 35]
    worst_class = [19]
    weights = 1.0 / np.sqrt(class_counts)
    for c in worst_classes:
        weights[c] *= 3.0 
    for c in worst_class:
        weights[c] *= 10.0 

    
    
    # test model met kleine dataset/ overfit test / full set
    if overfit_test:
        indices = torch.randperm(len(full_dataset), generator=g)[:500].tolist()
        test_train_dataset = Subset(full_dataset, indices)
    
        test_train_loader = DataLoader(test_train_dataset, batch_size=batch_size, shuffle=False,  num_workers=num_workers, persistent_workers= num_workers > 0, pin_memory=True, prefetch_factor=4)
        
        train_set = test_train_loader
        val_set = test_train_loader

    elif smal_test:
        train_indices_small = torch.randperm(len(full_dataset), generator=g)[:5000].tolist()
        test_train_dataset = Subset(full_dataset, train_indices_small)
        
        val_indices_small = torch.randperm(len(val_dataset), generator=g)[:5000].tolist()
        test_val_dataset = Subset(val_dataset, val_indices_small)
        
        test_indices_small = torch.randperm(len(test_dataset), generator=g).tolist()
        test_test_dataset = Subset(test_dataset, test_indices_small)
    
        train_sample_weights = [weights[labels[i]] for i in train_indices_small]
        sampler = WeightedRandomSampler(train_sample_weights, len(train_sample_weights))
    
        test_train_loader = DataLoader(test_train_dataset, batch_size=batch_size, sampler=sampler, shuffle=False, num_workers=num_workers, persistent_workers=num_workers > 0, pin_memory=True, prefetch_factor=4)
        test_val_loader   = DataLoader(test_val_dataset,   batch_size=batch_size, shuffle=False,          num_workers=num_workers, persistent_workers=num_workers > 0, pin_memory=True, prefetch_factor=4)
        test_test_loader  = DataLoader(test_test_dataset,  batch_size=batch_size, shuffle=False,          num_workers=num_workers, persistent_workers=num_workers > 0, pin_memory=True, prefetch_factor=4)
    
        train_set = test_train_loader
        val_set   = test_val_loader
        test_set  = test_test_loader
        
    else:
        n_total = len(full_dataset)
        n_val = int(n_total * val_frac)
        n_train = n_total - n_val


        indices = np.arange(len(full_dataset))
        
        train_indices, val_indices = train_test_split(
            indices,
            test_size=val_frac,
            stratify=labels,
            random_state=seed
        )
        
        train_dataset = Subset(full_dataset, train_indices)
        val_dataset = Subset(val_dataset, val_indices)
        
        train_sample_weights = [weights[labels[i]] for i in train_indices]
        sampler = WeightedRandomSampler(train_sample_weights, len(train_sample_weights))
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler,shuffle=False,  num_workers=num_workers, persistent_workers = num_workers > 0, pin_memory=True, prefetch_factor=2)
        val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, num_workers=num_workers, persistent_workers = num_workers > 0, pin_memory=True, prefetch_factor=2)
        test_loader  = DataLoader(test_dataset,   batch_size=batch_size, shuffle=False, num_workers=num_workers, persistent_workers = num_workers > 0, pin_memory=True, prefetch_factor=2)
        
        train_set = train_loader
        val_set   = val_loader
        test_set  = test_loader

    # sets 
    in_train_set = train_set
    in_val_set = val_set
    if overfit_test:
        in_test_set = in_train_set 
    else:
        in_test_set = test_set

    # inits
    
    model = CNN(in_channels=3, n_hidden=hidden_dims, n_classes=80, use_batch_norm=use_batch_norm,  use_se=use_se)
    
    class_weights_tensor = torch.FloatTensor(weights).to(device)
    loss_module = nn.CrossEntropyLoss(label_smoothing=0.1, weight=class_weights_tensor)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    if use_scheduler:
        warmup = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=5)
        cosine = CosineAnnealingLR(optimizer, T_max=epochs-5, eta_min=1e-5)
        scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[5])

    best_val = -1.0
    patience = 5          
    min_delta = 0.001     
    bad_epochs = 0
    best_val_loss = np.inf
    run_id = None

    # logging
    val_accuracies = []
    logs = dict()

    # inits training loop
    best_val_acc = -np.inf
    best_model = None
    model = model.to(device)
    scaler = torch.amp.GradScaler('cuda')
    
    try:
        for epoch in range(epochs):
            
            start = time.time()
            n_seen = 0
            running_loss=0
            correct = 0
            total = 0
            
            model.train()
            for x_batch, y_batch in tqdm(in_train_set):
                x_batch = x_batch.to(model.device, non_blocking=True)
                y_batch = y_batch.to(model.device, non_blocking=True)
            
                if np.random.rand() < 0.5:
                    x_mixed, y_a, y_b, lam = cutmix_batch(x_batch, y_batch, alpha=1.0)
                else:
                    x_mixed, y_a, y_b, lam = mixup_batch(x_batch, y_batch, alpha=0.2)

                
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    logits = model(x_mixed)
                    loss = lam * loss_module(logits, y_a) + (1-lam) * loss_module(logits, y_b)

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                
                bs = y_batch.size(0)
                running_loss += loss.item() * bs
                n_seen += bs
            
                preds = torch.argmax(logits, axis=1)
                total += len(preds)
                
                correct += (lam * (preds == y_a).float() + (1 - lam) * (preds == y_b).float()).sum()
            # print (train_acc, train_loss)
            train_acc = (correct / total).item()
            train_loss = running_loss / n_seen
            print(f"train_acc: {train_acc}, train_loss: {train_loss}")
    
            # print (val_acc, val_loss)
            val_acc, val_loss, all_preds, all_targets= evaluate_model(model, in_val_set, loss_module)
            print(f"val_acc: {val_acc}, val_loss: {val_loss}")
    
            # print (epoch time)
            epoch_time = time.time() - start
            print(f"Epoch {epoch+1} took {epoch_time:.2f} sec")
            
            if use_scheduler:
                scheduler.step()
                print(f"Epoch {epoch + 2}, Learning Rate: {scheduler.get_last_lr()[0]}")
                
            #logging
            logs[epoch + 1] = [train_acc, train_loss, val_acc, val_loss]
            val_accuracies.append(float(val_acc))
            
            if val_acc > best_val_acc:
                best_val_acc = val_acc

                checkpoint = {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_val_acc": best_val_acc,
                }
                
                if use_scheduler:
                    checkpoint["scheduler_state_dict"] = scheduler.state_dict()
                
                torch.save(checkpoint, "best_checkpoint.pt")
                best_model = deepcopy(model)
                print("Best checkpoint saved.")
                
            if val_loss < best_val_loss - min_delta:
                best_val_loss = val_loss
                bad_epochs = 0
            else:
                bad_epochs += 1
                        
            # early stopping with patience
            if bad_epochs >= patience and early_stopping:
                print("Early stopping triggered.")
                break
                
        if best_model is None:
            best_model = model
        test_model(best_model, in_test_set)
                        
        run_id = save_results(logs)
    except KeyboardInterrupt:
        print("Training handmatig gestopt.")
        if best_model is None:
            best_model = deepcopy(model)
        if logs:
            run_id = save_results(logs)
        return best_model, val_accuracies, logs, run_id, all_preds, all_targets
    return best_model, val_accuracies , logs, run_id, all_preds, all_targets

## test ensemble twee modelen

In [6]:
def test_ensemble(models, data_loader, n_tta=10):
    for model in models:
        model.eval()
    
    logits_accum = None
    all_names = []

    with torch.no_grad():
        for _ in range(n_tta):
            batch_logits = []
            for x_batch, name_batch in data_loader:
                x_batch = x_batch.to(models[0].device)
                # som van alle modellen
                logits = sum(model(x_batch) for model in models)
                batch_logits.append(logits)
                if _ == 0:
                    all_names.extend(name_batch)
            
            epoch_logits = torch.cat(batch_logits, dim=0)
            if logits_accum is None:
                logits_accum = epoch_logits
            else:
                logits_accum += epoch_logits

    preds = (torch.argmax(logits_accum, dim=1) + 1).cpu().tolist()
    df = pd.DataFrame({"img_name": all_names, "label": preds})
    df.to_csv("submission.csv", index=False)
    print("Saved submission.csv")

# 6. Experiment Configuration

Hyperparameters and paths for a single run.

In [7]:
# ---- hyperparameters ----
if torch.cuda.is_available():
    gc.collect()
    torch.cuda.empty_cache()

csv_path = "/kaggle/input/competitions/food-recognition-challenge-2026/train_labels.csv"
train_img_path = "/kaggle/input/competitions/food-recognition-challenge-2026/train_set/train_set/train_set"
test_img_path = "/kaggle/input/competitions/food-recognition-challenge-2026/test_set/test_set/test_set"

transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.03),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2),
])

base_config = {
    "hidden_dims": [64, 128, 256, 512, 512],
    "use_batch_norm": True,
    "lr": 1e-3,
    "batch_size": 128,
    "use_se": True,
    "epochs": 77,
    "val_frac": 0.1,
    "num_workers": 8,
    "overfit_test": False,
    "smal_test": False,
    "use_scheduler": True,
    "csv_path": csv_path,
    "train_img_path": train_img_path,
    "test_img_path": test_img_path,
    "early_stopping": False,
    "transform": transform,
    "weight_decay": 5e-4,
}
def run_training(seed, description):
    config = {**base_config, "seed": seed}
    
    best_model = None
    val_accuracies = []
    logs = {}
    run_id = None

    try:
        best_model, val_accuracies, logs, run_id, all_preds, all_targets = train(**config)
    except KeyboardInterrupt:
        print("\nTraining handmatig gestopt.")

    print("\n" + "="*50)
    print(f"TRAINING OVERVIEW - seed {seed}")
    print("="*50)
    print(f"Number of epochs: {len(val_accuracies)}")
    print("\nValidation accuracy per epoch:")
    for epoch_number, metrics in logs.items():
        print(f"epoch {epoch_number}: \n train_acc: {metrics[0]}, train_loss: {metrics[1]} \n val_acc: {metrics[2]}, val_loss: {metrics[3]}")

    if logs:
        best_epoch, best_metrics = max(logs.items(), key=lambda x: x[1][2])
        best_val_acc = best_metrics[2]
        print(f"\nBest validation accuracy: {best_val_acc:.4f} (at epoch {best_epoch})")

        run_summary = {
            "description": description,
            "run_id": run_id,
            **config,
            "best_val_acc": best_val_acc,
            "best_epoch": best_epoch,
        }
        run_summary["hidden_dims"] = json.dumps(run_summary["hidden_dims"])
        os.makedirs("experiments", exist_ok=True)
        summary_path = "experiments/experiment_summary.csv"
        df_row = pd.DataFrame([run_summary])
        write_header = not os.path.exists(summary_path)
        df_row.to_csv(summary_path, mode="a", header=write_header, index=False)
        print("="*50 + "\n")

        from sklearn.metrics import confusion_matrix
        cm = confusion_matrix(all_targets, all_preds)
        per_class_acc = cm.diagonal() / cm.sum(axis=1)
        for i in np.argsort(per_class_acc):
            print(f"class {i}: {per_class_acc[i]:.3f}")

    
    # model opslaan
    if best_model is not None:
        torch.save(best_model.state_dict(), f"model_seed{seed}.pt")
        print(f"Model opgeslagen als model_seed{seed}.pt")

    return best_model

# ---- Run 1: seed 42 ----
print("START RUN 3 (seed=456)")
model3 = run_training(seed=456, description="cutmix+warmup+SE seed456")

# zip experiments
import shutil
shutil.make_archive("experiments_backup", "zip", "experiments")
print("Klaar!")

START RUN 3 (seed=456)
DEVICE: cuda


  0%|          | 0/216 [00:00<?, ?it/s]

train_acc: 0.04191206395626068, train_loss: 3.0150831231534374
val_acc: 0.02416720986366272, val_loss: 4.383630269808679
Epoch 1 took 276.77 sec
Epoch 2, Learning Rate: 0.00028
Best checkpoint saved.


  0%|          | 0/216 [00:00<?, ?it/s]

train_acc: 0.05413447692990303, train_loss: 2.610973122764629
val_acc: 0.024820378050208092, val_loss: 4.386407995442017
Epoch 2 took 202.01 sec
Epoch 3, Learning Rate: 0.00045999999999999996
Best checkpoint saved.


...

train_acc: 0.5475364327430725, train_loss: 1.3101562270062372
val_acc: 0.5254735350608826, val_loss: 2.502001486285538
Epoch 75 took 202.91 sec
Epoch 76, Learning Rate: 1.1883624444585959e-05


  0%|          | 0/216 [00:00<?, ?it/s]

train_acc: 0.5417580604553223, train_loss: 1.311137621233987
val_acc: 0.5274330377578735, val_loss: 2.488104992797373
Epoch 76 took 203.02 sec
Epoch 77, Learning Rate: 1.0471130316980392e-05


  0%|          | 0/216 [00:00<?, ?it/s]

train_acc: 0.5406381487846375, train_loss: 1.3842545793536787
val_acc: 0.5258001089096069, val_loss: 2.506172823485481
Epoch 77 took 202.60 sec
Epoch 78, Learning Rate: 1e-05
Saved submission.csv

TRAINING OVERVIEW - seed 456
Number of epochs: 77

Validation accuracy per epoch:
epoch 1: 
 train_acc: 0.04191206395626068, train_loss: 3.0150831231534374 
 val_acc: 0.02416720986366272, val_loss: 4.383630269808679
epoch 2: 
 train_acc: 0.05413447692990303, train_loss: 2.610973122764629 
 val_acc: 0.024820378050208092, val_loss: 4.386407995442017
epoch 3: 
 train_acc: 0.06407074630260468, train_loss: 2.5922848586897667 
 val_acc: 0.03625081479549408, val_loss: 4.27828835382094
epoch 4: 
 train_acc: 0.0719568133354187, train_loss: 2.4756093908091854 
 val_acc: 0.037883736193180084, val_loss: 4.251995071720096
epoch 5: 
 train_acc: 0.08502300083637238, train_loss: 2.4122355805204916 
 val_acc: 0.06923579424619675, val_loss: 3.9302411745919814
epoch 6: 
 train_acc: 0.09003884345293045, train_lo

In [8]:
print("\nMaken ensemble submission...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

test_img_path = "/kaggle/input/competitions/food-recognition-challenge-2026/test_set/test_set/test_set"

test_dataset = FoodTestDataset(test_img_path)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False,
                         num_workers=8, persistent_workers=True,
                         pin_memory=True, prefetch_factor=2)

model1 = CNN(in_channels=3, n_hidden=[64,128,256,512,512], n_classes=80, use_batch_norm=True).to(device)
model1.load_state_dict(torch.load("/kaggle/input/models/janjunior/food-models/pytorch/default/3/model_seed42.pt", map_location=device))

model2 = CNN(in_channels=3, n_hidden=[64,128,256,512,512], n_classes=80, use_batch_norm=True).to(device)
model2.load_state_dict(torch.load("/kaggle/input/models/janjunior/food-models/pytorch/default/3/model_seed123.pt", map_location=device))

model3 = CNN(in_channels=3, n_hidden=[64,128,256,512,512], n_classes=80, use_batch_norm=True,  use_se=use_se).to(device)
model3.load_state_dict(torch.load("/kaggle/input/models/janjunior/food-models/pytorch/default/3/model_seed456.pt", map_location=device))

model1.eval()
model2.eval()
model3.eval()

test_ensemble([model1, model2, model3], test_loader, n_tta=20)
print("DONE")

Maken ensemble submission...
Saved submission.csv
